In [1]:
from astropy.io import fits
from scipy.io import readsav
import numpy as np
import os

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

import tools as t

In [2]:
data = torch.load("../data/PCA/TS1/t_0.pt")
data['rv']

/tmp/ipykernel_54544/3228788404.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load("../data/PCA/TS1/t_0.pt")


np.float64(0.0703277155342828)

In [3]:
# Parameters

N = 3686
M = 20
TS = 1

batch_size = 1
epochs = 1000

# Load data

X = []
y = []

for i in range(N):

    data = torch.load(f'../data/PCA/TS{TS}/t_{i}.pt')

    X.append(data['scores'].float())
    y.append(torch.as_tensor(data['rv'], dtype=torch.float32))

X = torch.stack(X)
y = torch.stack(y).reshape(-1, 1)

print("Total X shape:", X.shape)
print("Total y shape:", y.shape)

/tmp/ipykernel_54544/1867797996.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(f'../data/PCA/TS{TS}/t_{i}.pt')


Total X shape: torch.Size([3686, 20])
Total y shape: torch.Size([3686, 1])


In [4]:
# Chronological train/test split

train_size = int(0.8 * N)  # cutting at 80%
test_size = N - train_size

X_train = X[:train_size]
y_train = y[:train_size]

X_test = X[train_size:]
y_test = y[train_size:]

print("Train X shape:", X_train.shape)
print("Train y shape:", y_train.shape)

print("Test X shape:", X_test.shape)
print("Test y shape:", y_test.shape)

# Create Datasets

train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)


# Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

Train X shape: torch.Size([2948, 20])
Train y shape: torch.Size([2948, 1])
Test X shape: torch.Size([738, 20])
Test y shape: torch.Size([738, 1])


In [7]:
test_loader

In [5]:
# Model

# 1 layer
model = nn.Linear(M, 1)

# multi-layer
#model = nn.Sequential(
#    nn.Linear(M, 20),
#    nn.Linear(20, 10),
#    nn.Linear(10, 1)
#)

# Loss and optimizer

criterion = nn.MSELoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.01
)


# Training

for epoch in range(epochs):

    model.train()
    train_loss = 0.0

    for X_batch, y_batch in train_loader:

        # Forward pass
        y_pred = model(X_batch)

        # Calculate loss
        loss = criterion(y_pred, y_batch)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Average loss over batches
    train_loss /= len(train_loader)

    if epoch % 100 == 0 or epoch == epochs - 1:
        print(
            f"Epoch {epoch}: "
            f"train loss = {train_loss:.4f}"
        )

Epoch 0: train loss = 0.8785
Epoch 100: train loss = 0.8800
Epoch 200: train loss = 0.8801
Epoch 300: train loss = 0.8829
Epoch 400: train loss = 0.8831
Epoch 500: train loss = 0.8826
Epoch 600: train loss = 0.8809
Epoch 700: train loss = 0.8821
Epoch 800: train loss = 0.8798
Epoch 900: train loss = 0.8824
Epoch 999: train loss = 0.8796


In [6]:
# Testing

model.eval()

test_loss = 0.0

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        # Prediction
        y_pred = model(X_batch)

        # Loss
        loss = criterion(y_pred, y_batch)

        test_loss += loss.item()

# Average test loss
test_loss /= len(test_loader)

print(f"Test loss = {test_loss:.4f}")

Test loss = 0.9589
